# 21b — score ONE epoch of rung 21, against rung 18's SAME epoch

Run once per epoch (`-p EPOCH 1|2|3`), per arm (`-p ARM A_lr`). **Epoch-matched by
construction** (RULES §6b): the control is not "rung 18's best", it is rung 18's *same* epoch,
recomputed from its own archived answers through this same code path — never copied from a
table. Rungs 14 and 15 both compared their epoch 3 against rung 06's epoch 2 and one of them
recorded a win that lives entirely in that mismatch.

**The headline is the leaderboard proxy**, `mean(aggregation_ID, object_recognition_ID)`, not
`bucket_mean` — they are different quantities and must never be quoted against each other
(RULES §4b). `bucket_mean` and `margin_OOD` are reported beside it because the *final* ranking
is Copeland over buckets with ID and OOD weighted equally (RULES §4c), so an OOD loss is not
free just because the pre-eval cannot see it.

**Pre-registered:** a win requires the proxy to RISE **and** `margin_OOD` not to fall. A
faithful negative closes the "we are under-trained" hypothesis for one run, and that is a real
result.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, shutil, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT
# inherit the env's bin/ on PATH, and it must be THIS interpreter's bin.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models", EXP / "_tools",
          REPO / "experiments" / "18-count-aug" / "_models",
          REPO / "experiments" / "06-vit-lora" / "_models",
          REPO / "experiments" / "02-lora-sft" / "_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

# 🔴 HF_HOME must be set BEFORE the offline flags mean anything. The SDK's judge is cached at
# /workspace/hf_cache, NOT at the default ~/.cache/huggingface — which is empty on this pod.
# With OFFLINE set and HF_HOME unset, transformers looks in the empty default cache and raises
# from inside `run_baseline`, i.e. AFTER the 17 GB merge and the whole inference pass.
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.run import run_baseline
from recipe_sweep_train import RecipeSweepConfig, list_checkpoints, merge_checkpoint
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "| repo:", REPO)


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ----------
SMOKE = True          # True -> 40 questions, wiring only. Full: -p SMOKE False
EPOCH = 1             # 1 | 2 | 3 — resolved against list_checkpoints, not a step number
ARM   = "A_lr"        # A_lr | B_rank | control
RUN   = "21_lr_1e4_v1"

KEEP_MERGED = False   # a merged checkpoint is ~17 GB; keep only the one we ship
DATA_ROOT   = "/workspace/orena-data"

# The CONTROL: rung 18, the SAME epoch. Its per-epoch answers are archived, so every control
# number below is recomputed through this notebook's own code path rather than transcribed.
CONTROL_RUN = "/workspace/repo/experiments/18-count-aug/runs/18_count_aug_v1"


In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
RUN_DIR = EXP / "runs" / RUN
RUN_TAG = f"ep{EPOCH}_smoke" if SMOKE else f"ep{EPOCH}_full"

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING,
# and fail loudly: an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"

cfg = RecipeSweepConfig(exp_dir=EXP, run_name=RUN, data_root=DATA_ROOT)
CKPTS = list_checkpoints(cfg)
assert 1 <= EPOCH <= len(CKPTS), \
    f"EPOCH {EPOCH} but only {len(CKPTS)} checkpoints: {[c.name for c in CKPTS]}"
CKPT = CKPTS[EPOCH - 1]

# 🔴 The epoch-matched control. NOT rung 18's best epoch — rung 18's SAME epoch.
# 🔴 The epoch-matched control is the arm's OWN baseline, which is not always rung 18.
# Arm A won, so arms B and A2 are single variables off ARM A and must be read against IT:
# scoring arm B against rung 18 would price two changes (lr AND rank) as one.
from recipe_sweep_train import BASELINE_RUN
_baseline_run = BASELINE_RUN.get(ARM)
CTRL_RUN_DIR = Path(_baseline_run) if _baseline_run else Path(CONTROL_RUN)
CTRL_DIR = CTRL_RUN_DIR / f"ep{EPOCH}_full"
assert CTRL_DIR.is_dir() or SMOKE, (
    f"{CTRL_DIR} is missing — without rung 18's epoch {EPOCH} there is no control for this "
    "epoch, and an unevaluated epoch is a MISSING control, not a discarded one (RULES §6b)"
)

# The recipe this run ACTUALLY trained with, read from the args.json swift wrote beside the
# checkpoints. NOT from `cfg`: this notebook builds its config from the DEFAULTS (which are
# the control's values) because all it needs is a checkpoint path — so `cfg.learning_rate`
# is 2e-5 no matter which arm produced the weights. The first three rows this notebook ever
# wrote recorded lr=2e-05 for a run trained at 1e-4. Read the artifact, never the variable.
import glob as _glob
_args_json = sorted(_glob.glob(str(cfg.ckpt_dir / "*" / "args.json")))
assert _args_json, f"no args.json under {cfg.ckpt_dir} — cannot verify what was trained"
TRAINED = json.loads(Path(_args_json[-1]).read_text())
TRAINED = {k: TRAINED.get(k) for k in ("learning_rate", "lora_rank", "lora_alpha",
                                       "num_train_epochs", "seed",
                                       "per_device_train_batch_size",
                                       "gradient_accumulation_steps")}
_expected_lr = {"control": 2e-5, "A_lr": 1e-4, "A2_lr": 2e-4, "B_rank": 1e-4, "C_epochs": 2e-4, "D_clip": 2e-4, "A3_vitlr": 2e-4}[ARM]
_expected_rank = {"control": 8, "A_lr": 8, "A2_lr": 8, "B_rank": 32, "C_epochs": 8, "D_clip": 8, "A3_vitlr": 8}[ARM]
if abs(TRAINED["learning_rate"] - _expected_lr) > 1e-12 or TRAINED["lora_rank"] != _expected_rank:
    raise AssertionError(
        f"ARM={ARM!r} expects lr={_expected_lr} rank={_expected_rank}, but the checkpoint was "
        f"trained with {TRAINED} — the arm label and the weights disagree"
    )
if TRAINED["per_device_train_batch_size"] * TRAINED["gradient_accumulation_steps"] != 16:
    raise AssertionError(f"effective batch is not 16 in the trained run: {TRAINED}")

print(f"SMOKE   {SMOKE}   ARM {ARM}")
print(f"epoch   {EPOCH}/{len(CKPTS)} -> {CKPT.name}")
print(f"control {CTRL_DIR}   (baseline of arm {ARM})")
print(f"tag     {RUN_TAG}")
print(f"trained {TRAINED}")


In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ---
# `run_baseline` loads the judge only AFTER the 17 GB merge and the full inference pass, so a
# missing judge cache fails ~45 minutes in with everything already paid for. This is that
# failure, hoisted to the front and made cheap: the tokenizer alone proves the cache resolves
# under the offline flags. RAISES (RULES §7).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. The judge is cached at /workspace/hf_cache "
        "on this pod, not at the default ~/.cache/huggingface. Fix the env — do NOT disable "
        "the offline flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")


In [ ]:
# --- merge the adapter (per-epoch, namespaced so merges never overwrite) ---------
merged = cfg.merged_dir / CKPT.name
if merged.is_dir() and any(merged.iterdir()):
    print(f"OK    merged already present -> {merged}")
    MERGED_HERE = False
else:
    t0 = time.perf_counter()
    merged = merge_checkpoint(cfg, CKPT)
    MERGED_HERE = True
    print(f"      merged in {time.perf_counter() - t0:.0f}s -> {merged}")
merged


In [ ]:
# --- eval -----------------------------------------------------------------------
t0 = time.perf_counter()
try:
    cfg_eval = BaselineConfig(
        data_root=DATA_ROOT, model_path=merged, out_dir=RUN_DIR, run_name=RUN_TAG,
        max_pixels=1280 * 720, seed=42, n_eval=40 if SMOKE else None,
    )
    # The inference path must stay rung 06's EXACTLY. This rung's variable is the training
    # recipe; a post-processor or a second sample here would be a second variable and the
    # delta would stop being attributable.
    assert cfg_eval.answer_postprocess is None, "answer_postprocess must stay None"
    assert cfg_eval.n_samples == 1 and cfg_eval.enhance is None and cfg_eval.aux_view is None
    report = run_baseline(cfg_eval)
    print(f"eval done in {(time.perf_counter() - t0) / 60:.1f} min")
finally:
    if MERGED_HERE and not KEEP_MERGED and Path(merged).is_dir():
        shutil.rmtree(merged, ignore_errors=True)
        print(f"reclaimed ~17 GB -> removed {merged}")


In [ ]:
# --- canonical scoring + gates (all RAISE) --------------------------------------
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res = pd.read_csv(RUN_DIR / RUN_TAG / "results.csv")

missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
assert not missing, f"GATE 0 — {len(missing)} qIDs without gold; every margin would be inflated"

metrics.assert_no_dup_qid(res)
metrics.assert_ood_from_qid(res)
metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

# The mode gate, on the ARTIFACT rather than the variable: `-p SMOKE False` failing to take
# effect is otherwise indistinguishable from a successful full run.
assert (len(res) < 1000) == SMOKE, (
    f"MODE GATE FAILED: SMOKE={SMOKE} but the eval scored {len(res)} rows")
if not SMOKE:
    assert len(res) == 6252, f"expected the full eval set, got {len(res)} rows"

print("gates OK | bucket_mean:", round(strat["bucket_mean"], 4),
      "| margin_OOD:", round(strat["margin_OOD"], 4))


In [ ]:
# --- 🎯 THE HEADLINE: the leaderboard proxy, epoch-matched -----------------------
# mean(aggregation_ID, object_recognition_ID) — the quantity the platform actually scores on
# the pre-eval, where only those two buckets populate and both are ID (RULES §4b). It is NOT
# `bucket_mean`, and the two must never be quoted against each other.
def _proxy(report: dict) -> dict:
    # 🔴 A missing bucket is expected in SMOKE (a 40-row sample need not contain any
    # aggregation x ID question) and is a FINDING in a full run — so it returns NaN here and
    # the gate below raises only when the whole eval set was scored. The first smoke of this
    # notebook died on a bare .loc KeyError, which is the same failure with no diagnosis.
    bb = pd.DataFrame(report["by_bucket"])
    idc = bb[bb.distribution == "ID"].set_index("capability_group")
    out = {}
    for key, name in (("aggregation_ID", "aggregation"),
                      ("object_recognition_ID", "object_recognition")):
        out[key] = float(idc.loc[name, "accuracy"]) if name in idc.index else float("nan")
    out["proxy"] = (out["aggregation_ID"] + out["object_recognition_ID"]) / 2
    return out

arm_p = _proxy(strat)
if not SMOKE and any(pd.isna(v) for v in arm_p.values()):
    raise AssertionError(
        f"a scored bucket is missing from the full eval: {arm_p} — the leaderboard proxy is "
        "the mean of exactly these two, so a missing one is not a smaller number, it is no "
        "number at all"
    )

# The control is recomputed from rung 18's OWN archived answers through this same function —
# a transcribed number cannot be re-derived, and this one can.
ctrl_strat = None
if CTRL_DIR.is_dir() and (CTRL_DIR / "results.csv").exists():
    ctrl_res = pd.read_csv(CTRL_DIR / "results.csv")
    ctrl_strat = metrics.stratified_report(ctrl_res, gold=gold)
elif (CTRL_DIR / "stratified.json").exists():
    ctrl_strat = json.loads((CTRL_DIR / "stratified.json").read_text())

if ctrl_strat is None:
    print(f"⚠️  no control artifacts under {CTRL_DIR} — deltas below are NOT computed")
    ctrl_p = {k: float("nan") for k in arm_p}
else:
    ctrl_p = _proxy(ctrl_strat)

print(pd.DataFrame([
    {"cell": k, f"rung18_ep{EPOCH}": round(ctrl_p[k], 4), f"{ARM}_ep{EPOCH}": round(arm_p[k], 4),
     "delta": round(arm_p[k] - ctrl_p[k], 4)}
    for k in ("aggregation_ID", "object_recognition_ID", "proxy")
]).to_string(index=False))


In [ ]:
# --- 🆕 class-balanced F1 on `fo_class` — the metric the headline hides ----------
# More optimisation distance is most likely to do its damage in the TAIL, and exact-set
# accuracy cannot see the tail: rung 18 ep3 reads 0.6478 on fo_class x ID while `Gallstone`
# (n=28) recalls 0.036 and `External Drain` (n=24) 0.208.
preds = metrics.predictions_frame(RUN_DIR / RUN_TAG)
f1 = metrics.class_f1_report(preds, gold, results_df=res, n_boot=0 if SMOKE else 2000)

ctrl_f1 = None
if CTRL_DIR.is_dir() and (CTRL_DIR / "predictions.json").exists():
    ctrl_preds = metrics.predictions_frame(CTRL_DIR)
    ctrl_f1 = metrics.class_f1_report(ctrl_preds, gold, n_boot=0)

rows = []
for cell in ("pooled", "ID", "OOD"):
    b = f1[cell]
    a = ctrl_f1[cell] if ctrl_f1 else {"macro_f1": float("nan"), "exact_set_acc": float("nan")}
    rows.append({
        "cell": cell, "n": b["n"], "illegal": b["n_illegal"],
        "macro_f1_18": round(a["macro_f1"], 4), "macro_f1_arm": round(b["macro_f1"], 4),
        "d_macro": round(b["macro_f1"] - a["macro_f1"], 4),
        "exact_18": round(a["exact_set_acc"], 4), "exact_arm": round(b["exact_set_acc"], 4),
        "ci_arm": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]",
    })
f1_df = pd.DataFrame(rows)
print(f1_df.to_string(index=False))

# The per-class table is the whole point of this metric, so it is printed rather than stored —
# but a 40-question SMOKE need not contain one fo_class x ID row, and an empty frame has no
# columns to select. Absent in a FULL run it would be a finding, and the gate on the missing
# bucket has already raised by then.
_per = pd.DataFrame(f1["ID"]["per_class"]).T
print("\n--- per class, ID (the tail is the point) ---")
if _per.empty:
    print("   (no fo_class x ID rows in this sample — expected in SMOKE only)")
else:
    print(_per[["n_gold", "recall", "precision", "f1"]]
          .sort_values("n_gold", ascending=False).round(3).to_string())


In [ ]:
# --- count discrimination on the `Clips` template -------------------------------
# ⚠️ A rank correlation is a property of the SLICE, never of the model (RULES §13b): quote the
# template and n every time. And rank does NOT cash into score (§13c) — rung 06 ep3 orders OOD
# counts at r = 0.4997 while its `number` OOD margin is exactly 0.000000.
rank = metrics.count_rank_report(preds, gold)
ref_rank = (metrics.count_rank_report(metrics.predictions_frame(CTRL_DIR), gold)
            if CTRL_DIR.is_dir() and (CTRL_DIR / "predictions.json").exists() else None)

rows = []
for cell in ("pooled", "ID", "OOD"):
    b = rank[cell]
    a = ref_rank[cell] if ref_rank else {"r": float("nan"), "bias": float("nan"),
                                         "exact": float("nan")}
    rows.append({
        "cell": cell, "n": b["n"],
        f"r_18_ep{EPOCH}": round(a["r"], 4), f"r_{ARM}": round(b["r"], 4),
        "delta_r": round(b["r"] - a["r"], 4),
        "ci_arm": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]",
        "bias_18": round(a["bias"], 3), "bias_arm": round(b["bias"], 3),
        "unreadable_arm": b["n_unreadable"],
    })
rank_df = pd.DataFrame(rows)
print(rank_df.to_string(index=False))


In [ ]:
# --- the run's row + the PRE-REGISTERED verdict ---------------------------------
bf = pd.DataFrame(strat["by_format"])
num = bf[bf.answer_format == "number"].set_index("distribution")
ctrl_margin_ood = float(ctrl_strat["margin_OOD"]) if ctrl_strat else float("nan")

row = {
    "run": RUN, "arm": ARM, "epoch": EPOCH, "checkpoint": CKPT.name,
    "baseline_run": CTRL_RUN_DIR.name,
    "lr": TRAINED["learning_rate"], "lora_rank": TRAINED["lora_rank"],
    "lora_alpha": TRAINED["lora_alpha"],
    "proxy_leaderboard": arm_p["proxy"],
    "aggregation_ID": arm_p["aggregation_ID"],
    "object_recognition_ID": arm_p["object_recognition_ID"],
    "bucket_mean": strat["bucket_mean"],
    "acc_ID": strat["acc_ID"], "acc_OOD": strat["acc_OOD"],
    "margin_ID": strat["margin_ID"], "margin_OOD": strat["margin_OOD"],
    "number_margin_ID": num.loc["ID", "margin"] if "ID" in num.index else float("nan"),
    "number_margin_OOD": num.loc["OOD", "margin"] if "OOD" in num.index else float("nan"),
    "fo_class_macro_f1_ID": f1["ID"]["macro_f1"],
    "fo_class_macro_f1_OOD": f1["OOD"]["macro_f1"],
    "clips_r_pooled": rank["pooled"]["r"], "clips_bias": rank["pooled"]["bias"],
    "d_proxy_vs_baseline_same_epoch": arm_p["proxy"] - ctrl_p["proxy"],
    "d_margin_OOD_vs_baseline_same_epoch": strat["margin_OOD"] - ctrl_margin_ood,
}
print(pd.DataFrame([row]).T.to_string(header=False))

# 🔴 The pre-registered verdict, printed as a verdict and not left to the reader.
_proxy_up = row["d_proxy_vs_baseline_same_epoch"] > 0
_ood_held = row["d_margin_OOD_vs_baseline_same_epoch"] >= 0
print(f"\nPRE-REGISTERED READ (vs {CTRL_RUN_DIR.name} epoch {EPOCH})")
print(f"   leaderboard proxy rises : {_proxy_up}  "
      f"({row['d_proxy_vs_baseline_same_epoch']:+.4f})")
print(f"   margin_OOD does not fall: {_ood_held}  "
      f"({row['d_margin_OOD_vs_baseline_same_epoch']:+.4f})")
print(f"   -> {'WIN' if (_proxy_up and _ood_held) else 'NOT A WIN'}")
print("\n⚠️  There is NO seed-variance estimate in this project. A delta smaller than the "
      "paired video-clustered CI is not a result — quote the CI or do not quote the delta.")
print("A faithful negative is a real result and is recorded as one.")


In [ ]:
# --- persist: RESULTS.csv + the ledger (full runs only) -------------------------
if not SMOKE:
    # 🔴 One CSV PER ARM, not one shared file. Two pods share this network volume, so two
    # eval chains can be appending at the same moment -- and read-concat-write is not
    # atomic, so the later writer silently drops the earlier one's row. Per-arm files
    # cannot collide; `frame.ledger` and the reader merge them.
    out = EXP / f"RESULTS_{ARM}.csv"
    df = pd.concat([pd.read_csv(out), pd.DataFrame([row])], ignore_index=True) if out.exists() \
        else pd.DataFrame([row])
    df.to_csv(out, index=False)
    f1_df.to_csv(EXP / f"RESULTS_class_f1_{ARM}_ep{EPOCH}.csv", index=False)
    rank_df.to_csv(EXP / f"RESULTS_rank_{ARM}_ep{EPOCH}.csv", index=False)
    ledger.register_run(RUN_DIR / RUN_TAG, strat, experiment="21-recipe-sweep",
                        run=f"{RUN}__{RUN_TAG}",
                        model=f"21 {ARM} epoch {EPOCH} ({CKPT.name})", date="2026-07-28")
    print("wrote", out)
else:
    print("SMOKE — nothing persisted to RESULTS.csv or the ledger")


In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) -------
g = gold.copy()
g["template"] = g["question"].map(metrics.template_of)
clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
look = clips.merge(preds, on="qID").head(400)
look["gold_n"] = look["answer"].map(metrics.read_count)
look["pred_n"] = look["prediction"].map(metrics.read_count)
look["err"] = look["pred_n"] - look["gold_n"]

print("--- predicted-vs-gold crosstab (Clips) ---")
print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())
print("\n--- the worst counting misses ---")
print(look.reindex(look["err"].abs().sort_values(ascending=False).index)
          .head(12)[["qID", "gold_n", "pred_n", "prediction"]].to_string(index=False))

fo = res[res.answer_format == "fo_class"].merge(preds, on="qID")
wrong = fo[fo.correctness == 0].head(10)
print("\n--- fo_class misses (identity, not cardinality, is the usual failure) ---")
print(wrong[["qID", "prediction"]].to_string(index=False))


## After all three epochs

1. **Read the proxy per epoch**, epoch-matched. A win needs it to rise *and* `margin_OOD` not
   to fall, at the same epoch — the arm's epoch 3 against rung 18's epoch 3, never its best.
2. **Select by `acc_OOD`, per epoch** — never last-epoch-by-default (RULES §6). OOD is 50% of
   the final ranking.
3. **If arm A collapsed**, do not conclude "the recipe is wrong". The pre-registered
   diagnostic is arm **A2** — lr 1e-4 with `vit_lr` held at 2e-5 — because our LoRA reaches
   the ViT at the LLM's own LR and the Qwen3-VL default puts the tower 5–10× lower. Only A2
   separates "the recipe is wrong" from "the tower cannot take the LR".
4. **Only then launch arm B** (rank). If A won, B runs on top of A's learning rate and is a
   single variable off A, not off rung 18 — and must be reported that way.
5. Write the verdict to `context/decisions/` if it changes one, and regenerate the ledger.
